# 00 · Fuentes y descarga

**Pregunta del proyecto:** ¿cuánta gente camina cada octubre a Magdalena de Kino para las Fiestas de San Francisco Javier, y qué tan bien está medido ese fenómeno?

Este notebook es el inventario del proyecto: qué fuentes usamos, de dónde vienen, cómo se descargan y en qué estado están. No hace análisis; deja todo listo para los notebooks 01 a 08.

Hay dos tipos de datos:

| Tipo | Fuentes | Cómo se obtiene |
|---|---|---|
| **Datos abiertos** (descarga automática) | Censos y conteos INEGI 2000-2020, DENUE, proyección CONAPO/COESPO, clima Open-Meteo (ERA5), visitas de Wikipedia, rutas OSRM | `src/descarga.py` |
| **Registro manual de prensa** | Cifras de asistencia, derrama y distancias publicadas en medios de Sonora, con URL, fecha y declarante | `datos/manual/*.csv`, compilados a mano en la investigación documental (`research_notes/`) |

La segunda tabla existe porque **no hay ninguna fuente oficial de asistencia a las fiestas**. Ese hueco es, en sí mismo, uno de los hallazgos del proyecto.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.graph_objects as go

from src import config, tema
from src import procesamiento as proc

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Descarga

`src/descarga.py` es reanudable: si un archivo ya está en `datos/raw/`, no lo vuelve a bajar. Desde la terminal:

```bash
uv run python -m src.descarga                       # todo
uv run python -m src.descarga --solo clima rutas    # algunas fuentes
uv run python -m src.descarga --forzar --solo wikipedia
```

La celda siguiente descarga solo lo que falte.

In [2]:
import logging

from src import descarga

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
for nombre, funcion in descarga.FUENTES.items():
    funcion(forzar=False)

ITER 2000


  ya existe datos/raw/inegi/iter_26_2000.zip


ITER 2005


  ya existe datos/raw/inegi/iter_26_2005.zip


ITER 2010


...

  ya existe datos/raw/wikipedia/es.wikipedia__Eusebio_Francisco_Kino.json


Rutas OSRM


  ya existe datos/raw/rutas/osrm_rutas.json


Verificación de fuentes de prensa


  ya existe datos/raw/fuentes/estado_fuentes.csv


## 2. Inventario de archivos crudos

In [3]:
archivos = [
    {"archivo": str(p.relative_to(config.RAIZ)), "MB": round(p.stat().st_size / 1e6, 2)}
    for p in sorted(config.DIR_RAW.rglob("*"))
    if p.is_file() and p.suffix in {".zip", ".json", ".pdf", ".csv"} and "conjunto_de_datos" not in str(p)
]
pd.DataFrame(archivos)

,archivo,MB
0,datos/raw/clima/open_meteo_imuris.json,0.32
1,datos/raw/clima/open_meteo_magdalena.json,0.32
2,datos/raw/clima/open_meteo_nogales.json,0.32
3,datos/raw/coespo/Magdalena.pdf,1.41
4,datos/raw/denue/denue_26/diccionario_de_datos/...,0.01
5,datos/raw/denue/denue_26.zip,12.41
6,datos/raw/fuentes/estado_fuentes.csv,0.00
7,datos/raw/inegi/iter_26_2000/cgpv2000_iter_26/...,0.02
8,datos/raw/inegi/iter_26_2000.zip,0.50
9,datos/raw/inegi/iter_26_2005/cpv2005_iter_26/d...,0.03


## 3. Registro manual de cifras de prensa

Cada fila es **una cifra publicada**, no un año. Las columnas clave:

- `que_mide`: visitantes totales, caminantes, conteo parcial de Protección Civil, derrama, puestos.
- `momento`: si la cifra se dio **antes** del evento (proyección), **durante** (conteo parcial o estimación en curso) o **después** (balance).
- `verificacion`: `primaria` si leímos el texto original; `solo_titular` o `no_verificada` si no.

In [4]:
prensa = proc.leer_registro_prensa()
prensa[["id", "anio_evento", "fecha_publicacion", "medio", "cifra_min", "cifra_max", "unidad", "que_mide", "momento", "declarante", "verificacion"]]

,id,anio_evento,fecha_publicacion,medio,cifra_min,cifra_max,unidad,que_mide,momento,declarante,verificacion
0,P01,2020,2020-09-18,El Imparcial,140000.0,140000.0,personas,visitantes_totales,referencia_historica,NaN,primaria
1,P02,2022,2022-10-02,El Diario de Sonora,8000.0,8000.0,personas,caminantes,conteo_parcial,NaN,primaria
2,P03,2023,2023-10-10,El Imparcial,60000.0,60000.0,personas,visitantes_totales,balance,René Flores Rivera,primaria
3,P04,2023,2023-10-10,El Imparcial,60000000.0,60000000.0,pesos,derrama_economica,balance,René Flores Rivera,primaria
4,P05,2023,2023-10-10,El Imparcial,300.0,300.0,puestos,puestos_comerciales,balance,René Flores Rivera,primaria
5,P06,2024,2024-10-01,El Imparcial,45000.0,45000.0,personas,visitantes_totales,conteo_parcial,Ricardo Revilla Acuña,primaria
6,P07,2024,2024-10-01,El Imparcial,150000.0,150000.0,personas,visitantes_totales,proyeccion,Ricardo Revilla Acuña,primaria
7,P08,2024,NaT,Audiorama Comunicaciones,120000.0,120000.0,personas,visitantes_totales,proyeccion,NaN,solo_titular
8,P09,2025,2025-09-25,Expreso,12000.0,12000.0,personas,visitantes_totales,referencia_historica,Juan Francisco Gim,primaria
9,P10,2025,2025-09-30,Meganoticias,25000.0,25000.0,personas,caminantes,estimacion_en_curso,NaN,primaria


In [5]:
prensa.groupby(["verificacion"]).size().rename("cifras").to_frame()

,cifras
verificacion,
no_verificada,1
primaria,14
solo_titular,1


## 4. ¿Siguen vivas las fuentes?

Las notas de prensa desaparecen: el artículo de El Imparcial de 2019 sobre la peregrinación ya redirige a la portada. `descarga.verificar_fuentes` revisa el estado HTTP de cada URL citada y si existe copia en el Internet Archive.

In [6]:
estado = pd.read_csv(config.DIR_RAW / "fuentes" / "estado_fuentes.csv")
estado["dominio"] = estado["url"].str.extract(r"https?://(?:www\.)?([^/]+)")
estado[["dominio", "http", "copia_wayback"]].assign(copia_wayback=lambda d: d["copia_wayback"].notna())

,dominio,http,copia_wayback
0,eldiariodesonora.com.mx,200,False
1,eldiariodesonora.com.mx,200,False
2,eldiariodesonora.com.mx,200,False
3,audiorama.mx,error: SSLError,False
4,elimparcial.com,200,False
5,elimparcial.com,200,False
6,elimparcial.com,200,False
7,expreso.com.mx,200,False
8,meganoticias.mx,200,False
9,uniradioinforma.com,200,False


In [7]:
sin_copia = estado[estado["copia_wayback"].isna()]
print(f"{len(estado)} URLs citadas · {len(sin_copia)} sin copia en Internet Archive")

10 URLs citadas · 10 sin copia en Internet Archive


**Lectura:** las URLs sin copia archivada son las más frágiles del proyecto. Si una de ellas cae, la cifra solo queda respaldada por este registro. Conviene archivarlas manualmente en [web.archive.org/save](https://web.archive.org/save) antes de publicar.